# 3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

o find_nascentes quase que deu certo, mas tem um erros e antes de eu resolver esses erros eu preciso encontrar onde eles estõ acontecendo... bom, vamos começar usando um específico como exemplo, o do CORREGO MANDAQUI

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import os
from tqdm import tqdm


In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

# Raw gdf

In [3]:
drenageo = gpd.read_file(
    os.path.join(
        'data',
        'drenagem.zip'
    )
)

# Silver gdf

In [4]:
## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)
## Conferir se todos os 'cd_tipo_cu' sejam do mesmo tipo
drenageo['cd_tipo_cu'].dtype

dtype('float64')

# Determinar Correntes Estimadas (só por enquanto, dps vamos usar o do Elias e tals) 

In [5]:
drenageo.sample(10)
#* 11: trecho em estado natural
#* 12: lago ou reservatório
#* 10: trecho fechado
#* 9: trecho a céu aberto

cus_to_keep = [9.0, 11.0]
colors_dictionarie= {
    9.0 : 'turquoise',
    11.0 : 'aquamarine',
    10.0 : 'pink',
    12.0 : 'pink',
}

drenageo['colors'] = drenageo['cd_tipo_cu'].map(colors_dictionarie)

# Create GDF copy

In [6]:
gdf= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry'
]]

# Pontos buff

In [7]:
erro_pontos = gdf.copy()
erro_pontos_buff = gdf.copy()
erro_pontos['geometry']=shapely.get_point(gdf.geometry, -1)

erro_pontos_buff['geometry'] = erro_pontos['geometry'].buffer(10)

Que engraçado. Encontramos o erro e ele tá aqui em cima, não lá embaixo, onde eu esperava.

Minha teoria: o shapely só pega uma das extremidades por ponto, não duas, como eu acho que deveria ser o certo. 

In [8]:
teste=drenageo.loc[drenageo['cd_identif']==1649]
ponteste = shapely.get_point(teste.geometry, 0)
ponteste_fim = shapely.get_point(teste.geometry, -1)

In [9]:
teste

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry,colors
1771,1649.0,ND,None,1,VILA BARBOSA,SD,134.327921,11.0,Trecho em estado natural,JOAO BERNARDO DO COUTO,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (329536.818 7400628.483, 329540.219...",aquamarine


In [10]:
gdf_ponto = gpd.GeoDataFrame(geometry=ponteste)
gdf_ponto.set_crs(teste.crs, inplace=True)  # usa o CRS do original, se definido

gdf_ponfim = gpd.GeoDataFrame(geometry=ponteste_fim)
gdf_ponfim.set_crs(teste.crs, inplace=True)

,geometry
1771,POINT (329606.913 7400740.117)


# visualizar teste

m= teste.explore(color='orange')
gdf_ponto.explore(
    m=m
)

gdf_ponfim.explore(
    m=m,
    color='purple'
)

Descobrimos!!! Aaaah, o ponto é que se passar o `get_point()` com `0`, aí, vai retornar o primeiro ponto da linha e com o `-1` vai retornar o último ponto da linha... por isso que não estava dando certo.

# Correção

In [11]:
pontos = gdf.copy()
pontos_0 = gdf.copy()
pontos_1 = gdf.copy()
pontos_buff = gdf.copy()

pontos_0['geometry']=shapely.get_point(gdf.geometry, 0)
pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
pontos_1['geometry'] = shapely.get_point(gdf.geometry, -1)
pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"

pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)
pontos_buff['geometry'] = pontos['geometry'].buffer(10)

In [12]:
#E vamos, claro, conferir as intersecções
intersecs_bool=[]

for i, row in pontos_buff.iterrows():
    outras_geoms = pontos_buff.loc[pontos_buff.index!=i]
    outras_geoms.sample()
    intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            False
        )
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            True
        )

In [13]:
pontos_buff.sample()

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,intersec_bool
4248,3992.0,ND,9.0,SD,"POLYGON ((350932.738 7397218.589, 350932.689 7...",True


In [14]:
for i, row in pontos_buff.loc[pontos_buff['intersec_bool']==False].iterrows():
    outras_linhas = gdf.loc[gdf['cd_identif']!=row['cd_identif']]
    outras_linhas
    intersecs_bool = row.geometry.intersects(outras_linhas.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool']= False
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = True

In [15]:
pontos_buff.loc[pontos_buff['intersec_bool']!=False]

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,intersec_bool
1,26125.0,ND,11.0,SD,"POLYGON ((332074.747 7351146.304, 332074.699 7...",True
2,2.0,ND,11.0,SD,"POLYGON ((345331.612 7399298.312, 345331.564 7...",True
3,26126.0,COR,10.0,CORREGO PIRARUNGAUA,"POLYGON ((334522.87 7384823.742, 334522.822 73...",True
5,26152.0,ND,11.0,SD,"POLYGON ((320684.197 7386065.237, 320684.149 7...",True
6,26153.0,ND,10.0,SD,"POLYGON ((327674.456 7379012.447, 327674.408 7...",True
...,...,...,...,...,...,...
27603,27593.0,ND,12.0,SD,"POLYGON ((331058.55 7365876.449, 331058.502 73...",True
27604,27594.0,COR,11.0,CORREGO JACUPEVAL,"POLYGON ((350074.05 7396709.698, 350074.002 73...",True
27605,27595.0,ND,11.0,SD,"POLYGON ((340600.364 7412883.133, 340600.316 7...",True
27607,27597.0,ND,11.0,SD,"POLYGON ((322359.087 7346001.325, 322359.039 7...",True


In [16]:
pontos_buff.shape

(27611, 6)

In [17]:
pontos_buff['cd_tipo_cu'].astype(dtype='float', copy=False)
pontos_buff= pontos_buff.loc[pontos_buff['cd_tipo_cu'].isin(cus_to_keep)]
pontos_buff = pontos_buff.loc[pontos_buff['intersec_bool']==False]

In [18]:
pontos_buff.shape

(8279, 6)

# Visualizar
### Ok, eu estava errada... mesmo com o intersec e um mega buffer nos pontos, ainda não dá certo, vamos voltar pra tatica do Henrique mesmo
m= drenageo.explore(color='pink')
drenageo.loc[drenageo['nm_acident']=="CORREGO MANDAQUI"].explore(m=m, color='red')
pontos_buff.explore(
    m=m,
    color="green"
)
drenageo.loc[drenageo['cd_tipo_cu'].isin(cus_to_keep)].explore(
    m=m,
    color='purple'
)

E depois disso tudo, aquele bendito riozinho que o Mauryas achou ainda não foi pego! Vou fazer um commit pro que eu já fiz e depois passo as coreeções daqui lá pro find_nascentes, e depois procurar os motivos desse novo erro

In [21]:
# Salvar arquivo para a validação do Mauryas
pontos_buff.to_file(
    os.path.join(
        'data',
        'pontos_buff_v2.0.geojson'
    ),
    driver="GeoJSON"
)